# GA4 Web 用户增长与转化分析

```text
我用了 GA4 电商网站用户行为数据，先做字段清洗和口径定义，再分析浏览到购买漏斗、首次活跃留存、用户分层，最后给出业务建议和局限性。
```



## 1. 项目背景与分析目标

**项目背景：** 这是一个公开的 GA4 电商网站用户行为数据集，记录了用户访问、浏览商品、加购、结账和购买等事件。

**我想回答的问题：**

1. 用户从商品浏览到购买的过程中，哪一步流失最大？
2. 用户首次活跃后，Day 1 / Day 7 是否还会回来？
3. 不同活跃度、购买状态和渠道来源的用户质量有什么差异？
4. 这些结果应该如何转化成后续看板和业务建议？

**本 notebook 的分析主线：**

```text
读取数据 -> 清洗字段 -> 漏斗分析 -> 留存分析 -> 用户分层 -> 合理性检查 -> 结论和局限性
```


## 2. 最小口径卡


| 名称 | 一行代表什么 | 用途 |
|---|---|---|
| `raw_events_df` | 一次原始 GA4 用户事件 | 读取后的原始明细 |
| `clean_df` | 一次清洗后的用户事件 | 后续所有分析的基础表 |
| `funnel_summary` | 一个漏斗步骤 | 看每一步到达用户数和流失 |
| `retention_result` | 一个首次活跃 cohort 日期 | 看 Day 1 / Day 7 留存 |
| `user_features_df` | 一个用户 | 做活跃、购买和渠道分层 |
| `active_segment_summary` | 一个活跃度分层 | 对比低 / 中 / 高活跃用户质量 |
| `channel_segment_summary` | 一个用户首次 `source_medium` | 对比不同首次渠道来源的用户质量 |

关键指标口径：

| 指标 | 口径 |
|---|---|
| 用户数 | `user_pseudo_id` 去重 |
| `purchase_event_users` | 触发过 `purchase` 事件的去重用户，用来和漏斗 purchase 步骤对账 |
| `buyers / valid_order_users` | 有有效订单的去重用户，按 `valid_order_id` 判断 |
| 有效订单 | `event_name = purchase` 且 `transaction_id` 非空且不等于 `(not set)` |
| 收入 | `purchase_revenue_usd` 转成数值后汇总 |
| 相邻转化率 | 当前漏斗步骤用户数 / 上一步用户数 |
| Day N 留存率 | cohort 用户中第 N 天仍有事件的用户数 / cohort 用户数 |


## 3. 环境与路径设置


In [15]:
# 这格只做两件事：导入工具 + 定义读取时需要保留的字段。

import numpy as np
import pandas as pd
from IPython.display import display

# ID 字段是“编号”，不是数值指标，所以按字符串读取，避免变成科学计数法或 float。
ID_DTYPES = {
    'user_pseudo_id': 'string',
    'ga_session_id': 'string',
    'session_key': 'string',
    'transaction_id': 'string',
}

## 4. 读取数据

直接读取全量 CSV；这里先确认数据读进来了多少行、多少列、多少用户和多少事件类型。


In [16]:
raw_events_df = pd.read_csv(
    '/Users/wu/Downloads/课件及代码/数据分析师转行冲刺/projects/ga4_user_growth_analysis/data/interim/flat_events.csv',
    dtype=ID_DTYPES,
)

# 先做最基础的审计：读了多少行、多少列、多少用户、多少事件类型。
load_summary = pd.DataFrame([
    {'item': 'rows_loaded', 'value': len(raw_events_df)},
    {'item': 'columns_loaded', 'value': raw_events_df.shape[1]},
    {'item': 'unique_users_raw', 'value': raw_events_df['user_pseudo_id'].nunique()},
    {'item': 'event_types_raw', 'value': raw_events_df['event_name'].nunique()},
])

display(load_summary)
display(raw_events_df.head(3))


,item,value
0,rows_loaded,2742274
1,columns_loaded,23
2,unique_users_raw,270051
3,event_types_raw,13


,event_dt,event_time,user_pseudo_id,ga_session_id,session_key,event_name,platform,device_category,operating_system,device_language,...,user_source,user_medium,user_campaign,page_location,page_title,transaction_id,purchase_revenue,purchase_revenue_usd,total_item_quantity,unique_items
0,2020-11-23,2020-11-23 13:25:55.477657+00:00,10072324.1672857579,8718412163,10072324.1672857579-8718412163,first_visit,WEB,mobile,Web,NaN,...,google,cpc,<Other>,https://googlemerchandisestore.com/,Google Online Store,<NA>,NaN,NaN,NaN,NaN
1,2020-12-08,2020-12-08 00:21:34.477191+00:00,10169443.8944078183,3923939255,10169443.8944078183-3923939255,view_item,WEB,desktop,Web,en-us,...,google,organic,(organic),https://shop.googlemerchandisestore.com/Google...,Google Youth FC Tee Charcoal,(not set),NaN,NaN,NaN,NaN
2,2020-12-19,2020-12-19 06:36:29.787448+00:00,10258453.8220492341,2032603532,10258453.8220492341-2032603532,page_view,WEB,desktop,Macintosh,NaN,...,google,organic,(organic),https://shop.googlemerchandisestore.com/Google...,Google Stylus Pen w/ LED Light,<NA>,NaN,NaN,NaN,NaN


## 5. 清洗与基础审计

这一步的核心不是“把数据改漂亮”，而是把后面会用到的口径统一好。

重点记住 4 件事：

1. 时间字段要能做日期分析。
2. `purchase` 事件不等于有效订单，有效订单还要看 `transaction_id`。
3. 收入统一使用 `revenue_amount`，避免后面混用字段。
4. **后续漏斗、留存、用户分层都直接复用 `clean_df` 里已经清洗好的字段，不在每个模块重复清洗。**


In [17]:
# 基础清洗：把后面会用到的字段统一成稳定口径。
clean_df = raw_events_df.copy()

# 1. 时间字段转成 datetime，方便按日期、cohort 和留存计算。
clean_df['event_time'] = pd.to_datetime(clean_df['event_time'], errors='coerce', utc=True)
clean_df['event_dt'] = pd.to_datetime(clean_df['event_dt'], errors='coerce')
clean_df['event_date'] = clean_df['event_dt'].dt.normalize()

# 2. purchase 事件、有效订单、收入分别定义清楚。
# 注意：purchase 事件不一定等于有效订单；有效订单还要看 transaction_id 是否有效。
clean_df['is_purchase'] = clean_df['event_name'].eq('purchase')
clean_df['is_valid_order'] = (
    clean_df['is_purchase']
    & clean_df['transaction_id'].notna()
    & clean_df['transaction_id'].ne('(not set)')
)
clean_df['valid_order_id'] = clean_df['transaction_id'].where(clean_df['is_valid_order'])
clean_df['revenue_amount'] = pd.to_numeric(clean_df['purchase_revenue_usd'], errors='coerce').fillna(0)

# 3. 渠道字段缺失先标记为 unknown，不直接删除。
clean_df[['user_source', 'user_medium', 'user_campaign']] = (
    clean_df[['user_source', 'user_medium', 'user_campaign']]
    .fillna('unknown')
)

# 基础审计表：用于确认数据规模和购买口径。
basic_audit = pd.DataFrame([
    {'metric': 'events', 'value': len(clean_df)},
    {'metric': 'users', 'value': clean_df['user_pseudo_id'].nunique()},
    {'metric': 'date_count', 'value': clean_df['event_date'].nunique()},
    {'metric': 'event_type_count', 'value': clean_df['event_name'].nunique()},
    {'metric': 'min_event_date', 'value': clean_df['event_date'].min().date()},
    {'metric': 'max_event_date', 'value': clean_df['event_date'].max().date()},
    {'metric': 'purchase_events', 'value': int(clean_df['is_purchase'].sum())},
    {'metric': 'purchase_event_users', 'value': clean_df.loc[clean_df['is_purchase'], 'user_pseudo_id'].nunique()},
    {'metric': 'valid_order_users', 'value': clean_df.loc[clean_df['valid_order_id'].notna(), 'user_pseudo_id'].nunique()},
    {'metric': 'valid_orders', 'value': clean_df['valid_order_id'].nunique()},
    {'metric': 'total_revenue_usd', 'value': clean_df['revenue_amount'].sum()},
])

display(basic_audit)


,metric,value
0,events,2742274
1,users,270051
2,date_count,92
3,event_type_count,13
4,min_event_date,2020-11-01
5,max_event_date,2021-01-31
6,purchase_events,5692
7,purchase_event_users,4419
8,valid_order_users,3713
9,valid_orders,4451


## 6. 漏斗分析：商品浏览到购买

漏斗路径：

```text
view_item -> add_to_cart -> begin_checkout -> add_payment_info -> purchase
```

当前是 **步骤到达漏斗**：分别统计每一步触发过该事件的去重用户数。

这版可以回答：**哪个步骤人数下降最多？**

这版暂时不能回答：**每个用户是否严格按这个顺序完成路径？**


In [18]:
# 漏斗 v0：每一步分别统计“到达过这一步”的去重用户数。
# 这一版不校验严格顺序，所以叫“步骤到达漏斗”。
funnel_steps = pd.DataFrame([
    {'step_order': 1, 'step_name': '商品浏览', 'event_name': 'view_item'},
    {'step_order': 2, 'step_name': '加购', 'event_name': 'add_to_cart'},
    {'step_order': 3, 'step_name': '开始结账', 'event_name': 'begin_checkout'},
    {'step_order': 4, 'step_name': '支付信息', 'event_name': 'add_payment_info'},
    {'step_order': 5, 'step_name': '购买', 'event_name': 'purchase'},
])

funnel_summary = funnel_steps.copy()
funnel_summary['users'] = funnel_summary['event_name'].map(
    lambda event_name: clean_df.loc[clean_df['event_name'].eq(event_name), 'user_pseudo_id'].nunique()
)

# 相邻转化率：当前步骤用户数 / 上一步用户数。
# 整体转化率：当前步骤用户数 / 第一步商品浏览用户数。
funnel_summary['prev_users'] = funnel_summary['users'].shift(1)
funnel_summary['step_conversion_rate'] = funnel_summary['users'] / funnel_summary['prev_users']
funnel_summary['overall_conversion_rate'] = funnel_summary['users'] / funnel_summary['users'].iloc[0]
funnel_summary['drop_users_from_previous'] = (funnel_summary['prev_users'] - funnel_summary['users'])

display(funnel_summary)

# 找出相邻步骤里流失人数最多的一步。
biggest_drop = (
    funnel_summary
    .dropna(subset=['drop_users_from_previous'])
    .sort_values('drop_users_from_previous', ascending=False)
    .head(1)
)

display(biggest_drop)

,step_order,step_name,event_name,users,prev_users,step_conversion_rate,overall_conversion_rate,drop_users_from_previous
0,1,商品浏览,view_item,61252,NaN,NaN,1.000000,NaN
1,2,加购,add_to_cart,12545,61252.0,0.204810,0.204810,48707.0
2,3,开始结账,begin_checkout,9715,12545.0,0.774412,0.158607,2830.0
3,4,支付信息,add_payment_info,5751,9715.0,0.591971,0.093891,3964.0
4,5,购买,purchase,4419,5751.0,0.768388,0.072145,1332.0


,step_order,step_name,event_name,users,prev_users,step_conversion_rate,overall_conversion_rate,drop_users_from_previous
1,2,加购,add_to_cart,12545,61252.0,0.20481,0.20481,48707.0


## 7. 首次活跃 Cohort 留存

当前数据没有明确的注册事件，所以这里不做“注册留存”。

本项目使用：

```text
cohort_date = 用户第一次出现在数据里的日期
Day N 留存 = 这个用户在首次活跃后第 N 天仍然有任意事件
```


| 表 | 一行代表什么 |
|---|---|
| `user_cohort_df` | 一个用户的首次活跃日期 |
| `user_daily_active_df` | 一个用户在某一天活跃过 |
| `retention_result` | 一个 cohort 日期的 Day 1 / Day 7 留存 |


In [19]:
# 留存 v0：先定义每个用户的首次活跃日期，再看 Day 1 / Day 7 是否仍有事件。
# 当前没有注册事件，所以这里不是“注册留存”，而是“首次活跃 cohort 留存”。
# 这里直接复用 clean_df['event_date']，不再重复清洗时间字段。
retention_base_df = clean_df[['user_pseudo_id', 'event_date', 'event_name']].copy()

# 用户级 cohort 表：一行一个用户。
user_cohort_df = (
    retention_base_df
    .groupby('user_pseudo_id', as_index=False)
    .agg(cohort_date=('event_date', 'min'))
)

# 用户-日期活跃表：一行代表一个用户在某天活跃过。
user_daily_active_df = (
    retention_base_df[['user_pseudo_id', 'event_date']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# 合并后计算每次活跃距离首次活跃过了几天。
retention_activity_df = user_daily_active_df.merge(
    user_cohort_df,
    on='user_pseudo_id',
    how='inner'
)
retention_activity_df['retention_day'] = (
    retention_activity_df['event_date'] - retention_activity_df['cohort_date']
).dt.days
retention_activity_df = retention_activity_df[retention_activity_df['retention_day'] >= 0].copy()

# 分母：每个 cohort 日期的用户数。
cohort_users = (
    user_cohort_df
    .groupby('cohort_date', as_index=False)
    .agg(cohort_users=('user_pseudo_id', 'nunique'))
)

# 分子：Day 1 / Day 7 仍活跃的用户数。
day1_retention = (
    retention_activity_df[retention_activity_df['retention_day'].eq(1)]
    .groupby('cohort_date', as_index=False)
    .agg(day1_retained_users=('user_pseudo_id', 'nunique'))
)

day7_retention = (
    retention_activity_df[retention_activity_df['retention_day'].eq(7)]
    .groupby('cohort_date', as_index=False)
    .agg(day7_retained_users=('user_pseudo_id', 'nunique'))
)

retention_result = (
    cohort_users
    .merge(day1_retention, on='cohort_date', how='left')
    .merge(day7_retention, on='cohort_date', how='left')
)
retention_result[['day1_retained_users', 'day7_retained_users']] = (
    retention_result[['day1_retained_users', 'day7_retained_users']]
    .fillna(0)
    .astype(int)
)
retention_result['day1_retention_rate'] = retention_result['day1_retained_users'] / retention_result['cohort_users']
retention_result['day7_retention_rate'] = retention_result['day7_retained_users'] / retention_result['cohort_users']

# 最后几天没有完整 Day 7 观察窗口，不能直接拿来评价 Day 7 留存。
max_event_date = retention_base_df['event_date'].max()
retention_result['has_day1_window'] = retention_result['cohort_date'] <= max_event_date - pd.Timedelta(days=1)
retention_result['has_day7_window'] = retention_result['cohort_date'] <= max_event_date - pd.Timedelta(days=7)

valid_day1 = retention_result['has_day1_window']
valid_day7 = retention_result['has_day7_window']
weighted_day1_rate = retention_result.loc[valid_day1, 'day1_retained_users'].sum() / retention_result.loc[valid_day1, 'cohort_users'].sum()
weighted_day7_rate = retention_result.loc[valid_day7, 'day7_retained_users'].sum() / retention_result.loc[valid_day7, 'cohort_users'].sum()

retention_overview = pd.DataFrame([
    {'metric': 'cohort_count', 'value': len(retention_result)},
    {'metric': 'cohort_users_total', 'value': int(retention_result['cohort_users'].sum())},
    {'metric': 'cohorts_with_day7_window', 'value': int(valid_day7.sum())},
    {'metric': 'weighted_day1_retention_rate', 'value': weighted_day1_rate},
    {'metric': 'weighted_day7_retention_rate', 'value': weighted_day7_rate},
])

display(retention_overview)
display(retention_result.head(10))


,metric,value
0,cohort_count,92.000000
1,cohort_users_total,270051.000000
2,cohorts_with_day7_window,85.000000
3,weighted_day1_retention_rate,0.046166
4,weighted_day7_retention_rate,0.007844


,cohort_date,cohort_users,day1_retained_users,day7_retained_users,day1_retention_rate,day7_retention_rate,has_day1_window,has_day7_window
0,2020-11-01,2363,127,20,0.053745,0.008464,True,True
1,2020-11-02,3134,235,44,0.074984,0.014040,True,True
2,2020-11-03,4341,269,62,0.061967,0.014282,True,True
3,2020-11-04,3310,198,49,0.059819,0.014804,True,True
4,2020-11-05,2824,176,48,0.062323,0.016997,True,True
5,2020-11-06,2361,117,32,0.049555,0.013554,True,True
6,2020-11-07,1735,87,21,0.050144,0.012104,True,True
7,2020-11-08,1660,106,13,0.063855,0.007831,True,True
8,2020-11-09,2364,141,56,0.059645,0.023689,True,True
9,2020-11-10,2747,201,49,0.073171,0.017838,True,True


## 8. 用户分层

用户分层不是一开始就直接按明细表分组，而是先生成用户级特征表。

这一步的关键变化是：

```text
事件级 clean_df：一行 = 一次事件
用户级 user_features_df：一行 = 一个用户
```

这一节**只新增分层需要的行为标记**，例如是否浏览商品、是否加购。`event_date`、`revenue_amount`、`is_purchase`、`is_valid_order`、`valid_order_id` 都直接复用清洗阶段已经统一好的字段，不再重新判断一遍。


In [20]:
# 用户分层第一步：把事件级明细压成用户级特征表。
# 这一步的重点是：user_features_df 一行必须代表一个用户。
# 这里直接复用 clean_df 里已经清洗好的字段，不再重复判断有效订单或重新处理日期、收入。
segmentation_df = clean_df[
    [
        'user_pseudo_id', 'event_date', 'event_time', 'event_name',
        'user_source', 'user_medium', 'revenue_amount',
        'is_purchase', 'is_valid_order', 'valid_order_id'
    ]
].copy()

# 只新增用户分层需要的行为标记。
segmentation_df['is_view_item'] = segmentation_df['event_name'].eq('view_item')
segmentation_df['is_add_to_cart'] = segmentation_df['event_name'].eq('add_to_cart')

# 活跃特征：一个用户有多少天活跃、多少次事件、多少次浏览 / 加购。
user_activity_features_df = (
    segmentation_df
    .groupby('user_pseudo_id', as_index=False)
    .agg(
        first_active_date=('event_date', 'min'),
        last_active_date=('event_date', 'max'),
        active_days=('event_date', 'nunique'),
        event_count=('event_name', 'count'),
        view_item_count=('is_view_item', 'sum'),
        add_to_cart_count=('is_add_to_cart', 'sum'),
    )
)

# 购买特征：购买事件数、有效订单数、总收入。
# valid_order_id 是真实交易 ID；无效订单为缺失值。
# 因此 valid_order_count 用 valid_order_id 去重，而不是对 is_valid_order 求和。
user_purchase_features_df = (
    segmentation_df
    .groupby('user_pseudo_id', as_index=False)
    .agg(
        purchase_event_count=('is_purchase', 'sum'),
        valid_order_count=('valid_order_id', 'nunique'),
        total_revenue=('revenue_amount', 'sum'),
    )
)
user_purchase_features_df['is_buyer'] = user_purchase_features_df['valid_order_count'].gt(0)

# 渠道特征：用户首次渠道归因，取该用户最早一次事件的 user_source / user_medium。
user_channel_features_df = (
    segmentation_df
    .sort_values(['user_pseudo_id', 'event_time'])
    [['user_pseudo_id', 'user_source', 'user_medium']]
    .drop_duplicates(subset=['user_pseudo_id'], keep='first')
    .copy()
)

# 合并成最终用户级特征表。
user_features_df = (
    user_activity_features_df
    .merge(user_purchase_features_df, on='user_pseudo_id', how='left')
    .merge(user_channel_features_df, on='user_pseudo_id', how='left')
)

user_features_df[['purchase_event_count', 'valid_order_count', 'total_revenue']] = (
    user_features_df[['purchase_event_count', 'valid_order_count', 'total_revenue']]
    .fillna(0)
)
user_features_df['is_buyer'] = user_features_df['is_buyer'].fillna(False)
user_features_df[['user_source', 'user_medium']] = user_features_df[['user_source', 'user_medium']].fillna('unknown')

# 单次访问会产生多条事件，因此活跃度按活跃天数而不是 event_count 分层。
# 1 天 = 低活跃，2-3 天 = 中活跃，4 天及以上 = 高活跃。
user_features_df['active_segment'] = pd.cut(
    user_features_df['active_days'],
    bins=[0, 1, 3, np.inf],
    labels=['低活跃', '中活跃', '高活跃'],
    include_lowest=True
)
user_features_df['purchase_segment'] = np.where(user_features_df['is_buyer'], '已购买', '未购买')
# source_medium 是用户首次渠道，不是事件级渠道分组，也不是严格 session 归因。
user_features_df['source_medium'] = (
    user_features_df['user_source'].astype(str) + ' / ' + user_features_df['user_medium'].astype(str)
)

display(user_features_df.head(10))


,user_pseudo_id,first_active_date,last_active_date,active_days,event_count,view_item_count,add_to_cart_count,purchase_event_count,valid_order_count,total_revenue,is_buyer,user_source,user_medium,active_segment,purchase_segment,source_medium
0,10001363.4360935308,2020-12-12,2020-12-12,1,3,0,0,0,0,0.0,False,<Other>,organic,低活跃,未购买,<Other> / organic
1,1000223163.8035209215,2021-01-07,2021-01-07,1,4,0,0,0,0,0.0,False,(direct),(none),低活跃,未购买,(direct) / (none)
2,1000299.7413851356,2021-01-20,2021-01-20,1,4,0,0,0,0,0.0,False,(direct),(none),低活跃,未购买,(direct) / (none)
3,1000300.3223254235,2020-11-04,2020-11-04,1,5,0,0,0,0,0.0,False,<Other>,<Other>,低活跃,未购买,<Other> / <Other>
4,10003031.4607645453,2020-12-11,2020-12-11,1,3,0,0,0,0,0.0,False,(direct),(none),低活跃,未购买,(direct) / (none)
5,10004358.0897722689,2021-01-08,2021-01-08,1,4,0,0,0,0,0.0,False,google,cpc,低活跃,未购买,google / cpc
6,1000441.6293401995,2020-12-07,2020-12-07,1,10,0,0,0,0,0.0,False,google,organic,低活跃,未购买,google / organic
7,10005335.8064658740,2020-12-12,2020-12-19,3,26,3,0,0,0,0.0,False,google,cpc,中活跃,未购买,google / cpc
8,1000557.2911835023,2021-01-07,2021-01-07,1,8,0,0,0,0,0.0,False,<Other>,<Other>,低活跃,未购买,<Other> / <Other>
9,10006188.0272495437,2020-12-04,2020-12-04,1,3,0,0,0,0,0.0,False,<Other>,<Other>,低活跃,未购买,<Other> / <Other>


## 9. 分层结果汇总

这里把用户级表继续压成 summary 表，用来回答：不同用户群的用户数、购买率、订单数和人均收入有什么差异。


In [21]:
# 用户分层第二步：按分层字段汇总。
# 面试时重点讲结果表，不需要背函数。
def summarize_user_segment(df, segment_col, observed=False):
    summary = (
        df
        .groupby(segment_col, observed=observed)
        .agg(
            users=('user_pseudo_id', 'nunique'),
            buyers=('is_buyer', 'sum'),
            valid_orders=('valid_order_count', 'sum'),
            total_revenue=('total_revenue', 'sum'),
            avg_active_days=('active_days', 'mean'),
            avg_event_count=('event_count', 'mean'),
        )
        .reset_index()
    )
    summary['buyer_rate'] = summary['buyers'] / summary['users']
    summary['revenue_per_user'] = summary['total_revenue'] / summary['users']
    return summary

active_segment_summary = summarize_user_segment(user_features_df, 'active_segment', observed=False)
purchase_segment_summary = summarize_user_segment(user_features_df, 'purchase_segment')
channel_segment_summary = summarize_user_segment(user_features_df, 'source_medium').sort_values('users', ascending=False)

print('active_segment_summary')
display(active_segment_summary)

print('purchase_segment_summary')
display(purchase_segment_summary)

print('channel_segment_summary top 10 by users')
display(channel_segment_summary.head(10))


active_segment_summary


,active_segment,users,buyers,valid_orders,total_revenue,avg_active_days,avg_event_count,buyer_rate,revenue_per_user
0,低活跃,240502,1262,1338,91229.0,1.000000,7.243125,0.005247,0.379327
1,中活跃,25013,1477,1705,148387.0,2.206693,25.370767,0.059049,5.932395
2,高活跃,4536,974,1423,122549.0,5.080467,80.619268,0.214727,27.016975


purchase_segment_summary


,purchase_segment,users,buyers,valid_orders,total_revenue,avg_active_days,avg_event_count,buyer_rate,revenue_per_user
0,已购买,3713,3713,4466,335011.0,2.698896,105.467816,1.0,90.226501
1,未购买,266338,0,0,27154.0,1.159136,8.825898,0.0,0.101953


channel_segment_summary top 10 by users


,source_medium,users,buyers,valid_orders,total_revenue,avg_active_days,avg_event_count,buyer_rate,revenue_per_user
9,google / organic,93022,1226,1481,113013.0,1.170637,9.950356,0.013180,1.214906
2,(direct) / (none),64089,846,1021,83743.0,1.181342,10.207430,0.013200,1.306667
4,<Other> / <Other>,47095,582,679,52248.0,1.153328,9.637753,0.012358,1.109417
6,<Other> / referral,25245,341,394,33454.0,1.185462,10.154169,0.013508,1.325173
10,shop.googlemerchandisestore.com / referral,14745,264,320,28602.0,1.227467,11.187046,0.017904,1.939776
8,google / cpc,14279,181,205,18321.0,1.161426,9.705442,0.012676,1.283073
5,<Other> / organic,8700,110,140,10760.0,1.170345,9.918391,0.012644,1.236782
0,(data deleted) / (data deleted),2823,161,223,21915.0,1.754871,21.863974,0.057032,7.763018
3,<Other> / (data deleted),51,2,3,109.0,1.431373,13.098039,0.039216,2.137255
1,(data deleted) / referral,1,0,0,0.0,1.000000,35.000000,0.000000,0.000000


## 10. 合理性检查

这一节不是新的业务分析，而是检查前面的结果表有没有明显口径错误。

固定检查思路：

```text
用户级表是否一行一个用户
summary 表用户数加总是否能对回总用户数
购买率是否在 0 到 1 之间
收入是否非负
是否存在收入和订单口径不一致的异常现象
```


In [22]:
# 合理性检查：确认从事件级 -> 用户级 -> summary 表时，用户数没有丢失或被放大。
expected_user_count = clean_df['user_pseudo_id'].nunique()

quality_check_items = {
    'user_features_df rows equal unique users': len(user_features_df) == expected_user_count,
    'user_features_df unique users equal source users': user_features_df['user_pseudo_id'].nunique() == expected_user_count,
    'user_features_df has no duplicated users': user_features_df.duplicated('user_pseudo_id').sum() == 0,
    'active_segment_summary users sum matches': active_segment_summary['users'].sum() == expected_user_count,
    'purchase_segment_summary users sum matches': purchase_segment_summary['users'].sum() == expected_user_count,
    'channel_segment_summary users sum matches': channel_segment_summary['users'].sum() == expected_user_count,
    'buyers not greater than users': (
        (active_segment_summary['buyers'] <= active_segment_summary['users']).all()
        and (purchase_segment_summary['buyers'] <= purchase_segment_summary['users']).all()
        and (channel_segment_summary['buyers'] <= channel_segment_summary['users']).all()
    ),
    'buyer rates between 0 and 1': (
        active_segment_summary['buyer_rate'].between(0, 1).all()
        and purchase_segment_summary['buyer_rate'].between(0, 1).all()
        and channel_segment_summary['buyer_rate'].between(0, 1).all()
    ),
    'revenue metrics non-negative': (
        (active_segment_summary['total_revenue'] >= 0).all()
        and (purchase_segment_summary['total_revenue'] >= 0).all()
        and (channel_segment_summary['total_revenue'] >= 0).all()
        and (active_segment_summary['revenue_per_user'] >= 0).all()
        and (purchase_segment_summary['revenue_per_user'] >= 0).all()
        and (channel_segment_summary['revenue_per_user'] >= 0).all()
    ),
}

quality_check_df = pd.DataFrame({
    'check_item': list(quality_check_items.keys()),
    'passed': list(quality_check_items.values()),
})

# 额外记录一个数据质量现象：有收入但没有有效订单的用户。
# 这类情况不直接删除，而是在局限性中说明 transaction_id 和收入字段可能不完全一致。
non_buyer_revenue_users = user_features_df.loc[
    (~user_features_df['is_buyer']) & (user_features_df['total_revenue'] > 0),
    ['user_pseudo_id', 'purchase_event_count', 'valid_order_count', 'total_revenue']
]

print('expected_user_count:', expected_user_count)
print('failed_checks:', int((~quality_check_df['passed']).sum()))
print('non_buyer_positive_revenue_users:', len(non_buyer_revenue_users))

display(quality_check_df)
display(non_buyer_revenue_users.head(10))


expected_user_count: 270051
failed_checks: 0
non_buyer_positive_revenue_users: 353


,check_item,passed
0,user_features_df rows equal unique users,True
1,user_features_df unique users equal source users,True
2,user_features_df has no duplicated users,True
3,active_segment_summary users sum matches,True
4,purchase_segment_summary users sum matches,True
5,channel_segment_summary users sum matches,True
6,buyers not greater than users,True
7,buyer rates between 0 and 1,True
8,revenue metrics non-negative,True


,user_pseudo_id,purchase_event_count,valid_order_count,total_revenue
476,1014825.0200289249,1,0,183.0
925,10295267.8818269139,6,0,794.0
4134,11258187.3799854087,1,0,13.0
4367,11324968.3542327533,1,0,10.0
4446,11349260.6217093900,1,0,45.0
4705,11422508.1200893075,1,0,13.0
4891,1147529.6357293919,1,0,4.0
6267,1189643.6653312879,1,0,46.0
7508,1226476.1714273543,1,0,144.0
8013,12425101.9463695702,1,0,40.0


In [23]:
# 导出 Tableau 使用的 Python 分析结果表。
retention_result.to_csv(
    "../data/processed/retention_result.csv",
    index=False,
    encoding="utf-8"
)

active_segment_summary.to_csv(
    "../data/processed/active_segment_summary.csv",
    index=False,
    encoding="utf-8"
)

purchase_segment_summary.to_csv(
    "../data/processed/purchase_segment_summary.csv",
    index=False,
    encoding="utf-8"
)

## 11. 初步结论

### 结论 1：最大流失出现在商品浏览到加购

**数据证据：** `funnel_summary` 中，`view_item -> add_to_cart` 的流失人数最大。全量数据中的商品浏览用户数、加购用户数和流失人数见上方漏斗结果表，说明很多用户看了商品但没有进一步表达购买意向。


---

### 结论 2：首次活跃后的回访偏弱

**数据证据：** `retention_result` 输出了每个 cohort 日期的 Day 1 / Day 7 留存。当前全量数据加权 Day 1 留存和 Day 7 留存都偏低。


---

### 结论 3：低活跃用户规模最大，但不能只看人数决定优先级

**数据证据：** `active_segment_summary` 对比了低 / 中 / 高活跃用户的用户数、购买率、订单数和人均收入。低活跃用户规模最大，但购买表现较弱；中活跃用户购买率高于低活跃，高活跃用户的购买率和人均收入最高，但人数相对更少。



## 12. 局限性


| 限制 | 说明 | 下一步 |
|---|---|---|
| 数据范围限制 | 当前 notebook 默认读取全量；如果临时改为 sample 调试，必须重新标注样本限制 | MySQL / Tableau 继续使用全量口径对齐 |
| 漏斗限制 | v0 是步骤到达漏斗，不校验严格顺序 | 后续做严格顺序漏斗 |
| 留存限制 | 当前是首次活跃留存，不是注册留存 | 如果有注册事件，再做注册 cohort |
| 订单 / 收入限制 | 存在收入和有效交易 ID 不完全一致现象 | 后续做交易字段对账 |
| 渠道限制 | `channel_segment_summary` 使用用户首次渠道归因，不是事件级渠道分组，也不是严格 session 归因；`<Other>`、`(data deleted)` 不能解释成具体渠道 | 渠道结论只对可识别来源负责 |


